<a href="https://colab.research.google.com/github/adityab-tech/Prism/blob/main/Fine_Tuning_Kronos_Prism.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers accelerate peft huggingface_hub safetensors

In [2]:
!git clone https://github.com/shiyu-coder/Kronos.git

Cloning into 'Kronos'...
remote: Enumerating objects: 371, done.
remote: Total 371 (delta 0), reused 0 (delta 0), pack-reused 371 (from 1)
Receiving objects: 100% (371/371), 9.35 MiB | 40.38 MiB/s, done.
Resolving deltas: 100% (158/158), done.


In [28]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd Kronos

/content/Kronos


In [5]:
!ls

examples  finetune	LICENSE  README.md	   tests
figures   finetune_csv	model	 requirements.txt  webui


In [4]:
import os
print(os.getcwd())

/content/Kronos


In [6]:
!ls

examples  finetune	LICENSE  README.md	   tests
figures   finetune_csv	model	 requirements.txt  webui


In [7]:
!head -100 README.md

<div align="center">
  <h2><b>Kronos: A Foundation Model for the Language of Financial Markets </b></h2>
</div>


<div align="center">

</a> 
<a href="https://huggingface.co/NeoQuasar"> 
<img src="https://img.shields.io/badge/🤗-Hugging_Face-yellow" alt="Hugging Face"> 
</a> 
<a href="https://shiyu-coder.github.io/Kronos-demo/"> <img src="https://img.shields.io/badge/🚀-Live_Demo-brightgreen" alt="Live Demo"> </a>
<a href="https://github.com/shiyu-coder/Kronos/graphs/commit-activity"> 
<img src="https://img.shields.io/github/last-commit/shiyu-coder/Kronos?color=blue" alt="Last Commit"> 
</a> 
<a href="https://github.com/shiyu-coder/Kronos/stargazers"> 
<img src="https://img.shields.io/github/stars/shiyu-coder/Kronos?color=lightblue" alt="GitHub Stars"> 
</a> 
<a href="https://github.com/shiyu-coder/Kronos/network/members"> 
<img src="https://img.shields.io/github/forks/shiyu-coder/Kronos?color=yellow" alt="GitHub Forks"> 
</a> 
<a href="./LICENSE"> 
<img src="https://img.shields.io/githu

In [8]:
!ls model

__init__.py  kronos.py	module.py


In [9]:
!ls finetune

config.py   qlib_data_preprocess.py  train_predictor.py  utils
dataset.py  qlib_test.py	     train_tokenizer.py


In [10]:
!ls finetune_csv

config_loader.py  examples		  README_CN.md
configs		  finetune_base_model.py  README.md
data		  finetune_tokenizer.py   train_sequential.py


In [11]:
!sed -n '1,200p' finetune_csv/README.md

# Kronos Fine-tuning on Custom CSV Datasets

This module provides a comprehensive pipeline for fine-tuning Kronos models on your own CSV-formatted financial data. It supports both sequential training (tokenizer followed by predictor) and individual component training, with full distributed training capabilities.


## 1. Data Preparation

### Required Data Format

Your CSV file must contain the following columns:
- `timestamps`: DateTime stamps for each data point
- `open`: Opening price
- `high`: Highest price
- `low`: Lowest price  
- `close`: Closing price
- `volume`: Trading volume
- `amount`: Trading amount

(volume and amount can be 0 if not available)

### Sample Data Format

| timestamps | open | close | high | low | volume | amount |
|------------|------|-------|------|-----|--------|--------|
| 2019/11/26 9:35 | 182.45215 | 184.45215 | 184.95215 | 182.45215 | 15136000 | 0 |
| 2019/11/26 9:40 | 184.35215 | 183.85215 | 184.55215 | 183.45215 | 4433300 | 0 |
| 2019/11/26 9:45 | 18

In [12]:
!sed -n '1,250p' finetune_csv/finetune_base_model.py

import os
import sys
import json
import time
import pickle
import random
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
from time import gmtime, strftime
import logging
from logging.handlers import RotatingFileHandler
import datetime
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

sys.path.append('../')
from model import Kronos, KronosTokenizer, KronosPredictor
from config_loader import CustomFinetuneConfig


class CustomKlineDataset(Dataset):
    
    def __init__(self, data_path, data_type='train', lookback_window=90, predict_window=10, 
                 clip=5.0, seed=100, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15):
        self.data_path = data_path
        self.data_type = data_type
        self.lookback_window = lookback_window
        self.predict_window = predict_window
        self.window =

In [13]:
!sed -n '1,250p' finetune_csv/train_sequential.py

import os
import sys
import time
import argparse
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torch.distributed as dist

sys.path.append('../')
from model import Kronos, KronosTokenizer, KronosPredictor

from config_loader import CustomFinetuneConfig
from finetune_tokenizer import train_tokenizer, set_seed, setup_logging as setup_tokenizer_logging
from finetune_base_model import train_model, create_dataloaders, setup_logging as setup_basemodel_logging


class SequentialTrainer:
    
    def __init__(self, config_path: str = None):
        self.config = CustomFinetuneConfig(config_path)
        self.rank = int(os.environ.get("RANK", "0"))
        self.world_size = int(os.environ.get("WORLD_SIZE", "1"))
        self.local_rank = int(os.environ.get("LOCAL_RANK", str(self.config.device_id if hasattr(self.config, 'device_id') else 0)))
        self.device = self._setup_device()
        
        self.config.print_config_summary()
    
    def _setup_devic

In [14]:
!sed -n '1,250p' finetune_csv/config_loader.py

import os
import yaml
from typing import Dict, Any


class ConfigLoader:
    
    def __init__(self, config_path: str):

        self.config_path = config_path
        self.config = self._load_config()
    
    def _load_config(self) -> Dict[str, Any]:

        if not os.path.exists(self.config_path):
            raise FileNotFoundError(f"config file not found: {self.config_path}")
        
        with open(self.config_path, 'r', encoding='utf-8') as f:
            config = yaml.safe_load(f)
        
        config = self._resolve_dynamic_paths(config)
        
        return config
    
    def _resolve_dynamic_paths(self, config: Dict[str, Any]) -> Dict[str, Any]:

        exp_name = config.get('model_paths', {}).get('exp_name', '')
        if not exp_name:
            return config
        
        base_path = config.get('model_paths', {}).get('base_path', '')
        path_templates = {
            'base_save_path': f"{base_path}/{exp_name}",
            'finetuned_tokenizer': f"{b

In [15]:
import torch
import transformers
import huggingface_hub

print(torch.__version__)
print(transformers.__version__)
print(huggingface_hub.__version__)

2.11.0+cpu
5.12.1
1.20.1


In [23]:
!apt-get -qq install tree

In [25]:
!tree -L 2

.
├── examples
│   ├── get_akshare_date_2024-2025_x.py
│   ├── get_date_new.py
│   ├── prediction_akshare_2024-2025.py
│   ├── prediction_batch_example.py
│   ├── prediction_cn_markets_day.py
│   ├── prediction_example.py
│   ├── prediction_new_GUI.py
│   ├── prediction_new.py
│   ├── prediction_wo_vol_example.py
│   ├── run_backtest_kronos.py
│   └── yuce
├── figures
│   ├── backtest_result_example.png
│   ├── logo.png
│   ├── overview.png
│   └── prediction_example.png
├── finetune
│   ├── config.py
│   ├── dataset.py
│   ├── qlib_data_preprocess.py
│   ├── qlib_test.py
│   ├── train_predictor.py
│   ├── train_tokenizer.py
│   └── utils
├── finetune_csv
│   ├── config_loader.py
│   ├── configs
│   ├── data
│   ├── examples
│   ├── finetune_base_model.py
│   ├── finetune_tokenizer.py
│   ├── README_CN.md
│   ├── README.md
│   └── train_sequential.py
├── LICENSE
├── model
│   ├── __init__.py
│   ├── kronos.py
│   └── module.py
├── README.md
├── requirements.txt
├── tests
│   ├── data
│

In [38]:
important_folders = [
    "model",
    "finetune",
    "finetune_csv"
]

for folder in important_folders:
    print(folder)
    print(os.listdir(folder))

model
['kronos.py', '__init__.py', 'module.py']
finetune
['qlib_test.py', 'train_tokenizer.py', 'config.py', 'utils', 'qlib_data_preprocess.py', 'train_predictor.py', 'dataset.py']
finetune_csv
['train_sequential.py', 'finetune_base_model.py', '__pycache__', 'examples', 'finetune_tokenizer.py', 'README.md', 'config_loader.py', 'data', 'README_CN.md', 'configs']


In [27]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [31]:
#Repository Structure

## model/
#- Contains the core Kronos model architecture.
#- Includes the Transformer blocks, tokenizer implementation, and model modules.
#- This is where the pretrained Kronos model is defined.

## finetune/
#- Contains the original fine-tuning pipeline provided by the authors.
#- Used for training the tokenizer and predictor separately.

## finetune_csv/
#- Contains the CSV-based fine-tuning pipeline.
#- This is the folder we will use for the PRISM project because our stock data is stored in CSV files.
#- Includes training scripts, configuration files, and dataset loading utilities.

## configs/
#- Stores YAML configuration files.
#- Defines parameters such as lookback window, prediction window, batch size, learning rate, and data paths.

## data/
#- Stores the input CSV files used for training.
#- For our project, we will replace these with Indian stock CSVs.

## train_sequential.py
#- Main script that performs sequential training.
#- It trains the tokenizer first and then the Kronos predictor.

In [32]:
!ls finetune_csv/configs

config_ali09988_candle-5min.yaml


In [33]:
!sed -n '1,200p' finetune_csv/configs/config_ali09988_candle-5min.yaml

#This is a template config for custom finetuning kronos on csv data
#这是一份模板config，用于kronos的csv自定义数据微调

data:
  data_path: "/xxxx/Kronos/finetune_csv/data/HK_ali_09988_kline_5min_all.csv"
  lookback_window: 512
  predict_window: 48
  max_context: 512
  clip: 5.0
  # dataset split ratio
  train_ratio: 0.9
  val_ratio: 0.1
  test_ratio: 0.0

training:
  # control the training epochs of tokenizer and basemodel
  tokenizer_epochs: 30
  basemodel_epochs: 20
  batch_size: 32
  log_interval: 50
  num_workers: 6
  seed: 42
  
  tokenizer_learning_rate: 0.0002
  predictor_learning_rate: 0.000001
  
  adam_beta1: 0.9
  adam_beta2: 0.95
  adam_weight_decay: 0.1
  
  # gradient accumulation steps for tokenizer training
  accumulation_steps: 1

# model path configuration
model_paths:
  # pretrained model path
  pretrained_tokenizer: "/xxx/Kronos/pretrained/Kronos-Tokenizer-base"
  pretrained_predictor: "/xxx/Kronos/pretrained/Kronos-base"
  
  # experiment name - other paths will be generated based 

In [35]:
from finetune_csv.config_loader import CustomFinetuneConfig

In [36]:
config = CustomFinetuneConfig(
    "finetune_csv/configs/config_ali09988_candle-5min.yaml")

In [37]:
config.print_config_summary()

Kronos finetuning configuration summary
Experiment name: HK_ali_09988_kline_5min_all
Data path: /xxxx/Kronos/finetune_csv/data/HK_ali_09988_kline_5min_all.csv
Lookback window: 512
Predict window: 48
Tokenizer training epochs: 30
Basemodel training epochs: 20
Batch size: 32
Tokenizer learning rate: 0.0002
Predictor learning rate: 1e-06
Train tokenizer: True
Train basemodel: True
Skip existing: False
Use pre-trained tokenizer: True
Use pre-trained predictor: True
Base save path: /xxx/Kronos/finetune_csv/finetuned//HK_ali_09988_kline_5min_all
Tokenizer save path: /xxx/Kronos/finetune_csv/finetuned//HK_ali_09988_kline_5min_all/tokenizer
Basemodel save path: /xxx/Kronos/finetune_csv/finetuned//HK_ali_09988_kline_5min_all/basemodel


In [40]:
#upar jo output mai given hai bass usse wapas print karwa raha hu taaki easy dikhe lol nigga
print("Data Path :", config.data_path)
print("Lookback :", config.lookback_window)
print("Predict :", config.predict_window)
print("Batch Size :", config.batch_size)
print("Tokenizer LR :", config.tokenizer_learning_rate)
print("Predictor LR :", config.predictor_learning_rate)

Data Path : /xxxx/Kronos/finetune_csv/data/HK_ali_09988_kline_5min_all.csv
Lookback : 512
Predict : 48
Batch Size : 32
Tokenizer LR : 0.0002
Predictor LR : 1e-06


In [41]:
#lookback_window -> Number of past candles.
#predict_window-> Future candles.
#batch_size -> Training batch.
#pretrained_predictor_path->Kronos-base.
#pretrained_tokenizer_path->Tokenizer.

In [42]:
!sed -n '1,250p' finetune/dataset.py

import pickle
import random
import numpy as np
import torch
from torch.utils.data import Dataset
from config import Config


class QlibDataset(Dataset):
    """
    A PyTorch Dataset for handling Qlib financial time series data.

    This dataset pre-computes all possible start indices for sliding windows
    and then randomly samples from them during training/validation.

    Args:
        data_type (str): The type of dataset to load, either 'train' or 'val'.

    Raises:
        ValueError: If `data_type` is not 'train' or 'val'.
    """

    def __init__(self, data_type: str = 'train'):
        self.config = Config()
        if data_type not in ['train', 'val']:
            raise ValueError("data_type must be 'train' or 'val'")
        self.data_type = data_type

        # Use a dedicated random number generator for sampling to avoid
        # interfering with other random processes (e.g., in model initialization).
        self.py_rng = random.Random(self.config.seed)

        # Set

In [43]:
!grep -n "class" finetune/dataset.py

9:class QlibDataset(Dataset):


In [44]:
!grep -n "__getitem__" finetune/dataset.py
!grep -n "__len__" finetune/dataset.py

92:    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
88:    def __len__(self) -> int:


In [45]:
#CSV ->Read using pandas-> Sliding Window ->Lookback Window ->Predict Window ->Dataset ->DataLoader

In [46]:
import os
data_path = "finetune_csv/data"
print(os.listdir(data_path))

['HK_ali_09988_kline_5min_all.csv']


In [47]:
sample_df = pd.read_csv(
    "finetune_csv/data/HK_ali_09988_kline_5min_all.csv")
sample_df.head()

,timestamps,open,close,high,low,volume,amount
0,2019/11/26 9:35,182.45215,184.45215,184.95215,182.45215,15136000,0
1,2019/11/26 9:40,184.35215,183.85215,184.55215,183.45215,4433300,0
2,2019/11/26 9:45,183.85215,183.35215,183.95215,182.95215,3070900,0
3,2019/11/26 9:50,183.35215,183.75215,183.95215,183.25215,2288600,0
4,2019/11/26 9:55,183.85215,183.85215,184.15215,183.75215,1770000,0


In [48]:
print(sample_df.columns.tolist())

['timestamps', 'open', 'close', 'high', 'low', 'volume', 'amount']


In [49]:
sample_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93912 entries, 0 to 93911
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   timestamps  93912 non-null  object 
 1   open        93912 non-null  float64
 2   close       93912 non-null  float64
 3   high        93912 non-null  float64
 4   low         93912 non-null  float64
 5   volume      93912 non-null  int64  
 6   amount      93912 non-null  int64  
dtypes: float64(4), int64(2), object(1)
memory usage: 5.0+ MB


In [50]:
my_df = pd.read_csv(
    "/content/drive/MyDrive/PRISM/raw_data/TCS.NS.csv"
)
print("My Dataset Columns:")
print(my_df.columns.tolist())

print("\nKronos Dataset Columns:")
print(sample_df.columns.tolist())

My Dataset Columns:
['Date', 'Close', 'High', 'Low', 'Open', 'Volume']

Kronos Dataset Columns:
['timestamps', 'open', 'close', 'high', 'low', 'volume', 'amount']


In [53]:
sample_df["timestamps"] = pd.to_datetime(sample_df["timestamps"])
my_df["Date"] = pd.to_datetime(my_df["Date"])